In [110]:
import json
import os
from pathlib import Path
from statistics import mean, stdev

# =============================================================================
# Configuration
# =============================================================================
FOLDER_PATH = "./raw_tree"
OUTPUT_FILE = "evaluation_report.txt"

STATS_TO_AVERAGE = [
    "total_nodes",
    "api_calls",
    "solutions_found",
    "code_executions",
    "code_errors",
    "deadend_memory_skipped",
    "total_input_tokens",
    "total_output_tokens",
    "api_saved_two_num_cache"
]

# Thresholds for detecting incomplete runs (API failures)
INCOMPLETE_THRESHOLDS = {
    "max_total_nodes": 2,          # only root, or root+1 child
    "max_api_calls": 2,            # very few API calls
    "max_code_executions": 0,      # no code executed
}

# Low‑activity flag: flag if total_nodes OR api_calls is below mean - (STD_DEV_MULTIPLIER * std)
STD_DEV_MULTIPLIER = 1.0   # 1 standard deviation below mean

# =============================================================================
# Process a single JSON file
# =============================================================================
def process_file(filepath: Path):
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    metadata = data.get("metadata", {})
    stats = metadata.get("statistics", {})

    solutions_found = stats.get("solutions_found", 0)
    success = solutions_found > 0

    total_nodes = stats.get("total_nodes", 0)
    api_calls = stats.get("api_calls", 0)
    code_executions = stats.get("code_executions", 0)

    # Detect incomplete run (API failure)
    incomplete = False
    if (total_nodes <= INCOMPLETE_THRESHOLDS["max_total_nodes"] and
        api_calls <= INCOMPLETE_THRESHOLDS["max_api_calls"] and
        code_executions <= INCOMPLETE_THRESHOLDS["max_code_executions"]):
        incomplete = True

    return {
        "file": filepath.name,
        "stats": stats,
        "solutions_found": solutions_found,
        "success": success,
        "incomplete": incomplete,
        "total_nodes": total_nodes,
        "api_calls": api_calls,
        "code_executions": code_executions,
    }

# =============================================================================
# Main aggregation and reporting
# =============================================================================
def main():
    folder = Path(FOLDER_PATH)
    if not folder.exists():
        print(f"Folder '{FOLDER_PATH}' does not exist.")
        return

    json_files = list(folder.glob("*.json"))
    if not json_files:
        print(f"No JSON files found in '{FOLDER_PATH}'.")
        return

    results = []
    for f in json_files:
        try:
            res = process_file(f)
            results.append(res)
        except Exception as e:
            print(f"Error processing {f.name}: {e}")

    total_files = len(results)
    successful_files = sum(1 for r in results if r["success"])
    success_rate = successful_files / total_files if total_files > 0 else 0.0

    incomplete_files = [r for r in results if r["incomplete"]]
    unsolved_but_complete = [r for r in results if not r["success"] and not r["incomplete"]]

    # Compute averages and standard deviations
    averages = {}
    std_devs = {}
    for key in STATS_TO_AVERAGE:
        values = [r["stats"].get(key, 0) for r in results]
        avg = mean(values) if values else 0.0
        averages[key] = avg
        if len(values) > 1:
            std_devs[key] = stdev(values) if values else 0.0
        else:
            std_devs[key] = 0.0

    # Low‑activity flagging (below mean - 1 std) – EXCLUDE incomplete AND successful files
    low_activity_files = []
    for r in results:
        if r["incomplete"] or r["success"]:
            continue   # skip both failures and successes
        reasons = []
        nodes = r["total_nodes"]
        api = r["api_calls"]
        nodes_avg = averages.get("total_nodes", 0)
        nodes_std = std_devs.get("total_nodes", 0)
        api_avg = averages.get("api_calls", 0)
        api_std = std_devs.get("api_calls", 0)

        if nodes_std > 0 and nodes < nodes_avg - STD_DEV_MULTIPLIER * nodes_std:
            reasons.append(f"total_nodes={nodes} < {nodes_avg:.1f} - {STD_DEV_MULTIPLIER}*std")
        if api_std > 0 and api < api_avg - STD_DEV_MULTIPLIER * api_std:
            reasons.append(f"api_calls={api} < {api_avg:.1f} - {STD_DEV_MULTIPLIER}*std")
        if reasons:
            low_activity_files.append((r["file"], reasons, nodes, api))

    # Write report
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as out:
        out.write("=" * 70 + "\n")
        out.write("GAME 24 TREE SEARCH EVALUATION REPORT\n")
        out.write("=" * 70 + "\n\n")

        out.write(f"Total JSON files processed: {total_files}\n")
        out.write(f"Files with solutions found: {successful_files}\n")
        out.write(f"Success rate: {success_rate:.2%}\n\n")

        out.write(f"INCOMPLETE RUNS (API failures): {len(incomplete_files)}\n")
        out.write(f"Unsolved but complete runs: {len(unsolved_but_complete)}\n\n")

        out.write("-" * 50 + "\n")
        out.write("AVERAGE STATISTICS (across all files)\n")
        out.write("-" * 50 + "\n")
        for key in STATS_TO_AVERAGE:
            out.write(f"{key:25s}: {averages[key]:10.2f}  (std: {std_devs[key]:.2f})\n")

        out.write("\n" + "-" * 50 + "\n")
        out.write("INCOMPLETE RUNS (likely API failures)\n")
        out.write("-" * 50 + "\n")
        if incomplete_files:
            for r in incomplete_files:
                out.write(f"{r['file']:50s} nodes={r['total_nodes']}, api_calls={r['api_calls']}, code_exec={r['code_executions']}\n")
        else:
            out.write("None detected.\n")

        out.write("\n" + "-" * 50 + "\n")
        out.write("LOW‑ACTIVITY RUNS (unsolved, but nodes/api_calls far below avg – check for poor proposals or early stop)\n")
        out.write("-" * 50 + "\n")
        if low_activity_files:
            for file_name, reasons, nodes, api in low_activity_files:
                out.write(f"{file_name:50s} nodes={nodes}, api_calls={api}\n")
                for reason in reasons:
                    out.write(f"    → {reason}\n")
        else:
            out.write("None detected.\n")

        out.write("\n" + "-" * 50 + "\n")
        out.write("UNSOLVED BUT COMPLETE RUNS (valid failures, not low‑activity)\n")
        out.write("-" * 50 + "\n")
        # Show unsolved runs that were NOT flagged as low‑activity
        unsolved_not_low = [r for r in unsolved_but_complete 
                            if not any(f[0] == r["file"] for f in low_activity_files)]
        if unsolved_not_low:
            for r in unsolved_not_low:
                out.write(f"{r['file']:50s} nodes={r['total_nodes']}, api_calls={r['api_calls']}\n")
        else:
            out.write("None.\n")

        out.write("\n" + "-" * 50 + "\n")
        out.write("DETAILED FILE RESULTS\n")
        out.write("-" * 50 + "\n")
        for r in results:
            if r["success"]:
                status = "SOLUTION"
            elif r["incomplete"]:
                status = "INCOMPLETE"
            else:
                status = "UNSOLVED"
            out.write(f"{r['file']:50s} {status}\n")

    print(f"Report written to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Report written to evaluation_report.txt
